# 🌍 Spatial Asset Detection — Land Cover Training Pipeline

**Model:** YOLOv11m-seg (Instance Segmentation)  
**Dataset:** [DeepGlobe Land Cover Classification](https://www.kaggle.com/datasets/balraj98/deepglobe-land-cover-classification-dataset)  
**Classes:** Trees & Green Cover, Parks & Open Spaces, Water Bodies, Roads & Footpaths  

This notebook converts DeepGlobe's RGB semantic masks into YOLO polygon format
and trains a segmentation model optimized for satellite/aerial imagery.

### How to run
1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**
3. Trained weights (`best.pt`) will auto-download when finished

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 1: Install Dependencies                   ║
# ╚══════════════════════════════════════════════════╝
!pip install -U ultralytics==8.3.0 opencv-python-headless kaggle

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [2]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 2: Download DeepGlobe Dataset from Kaggle ║
# ╚══════════════════════════════════════════════════╝
import os

os.environ['KAGGLE_API_TOKEN'] = 'KGAT_fdddba790fc55f60fa5518ae997a5811'
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as f:
    f.write(os.environ['KAGGLE_API_TOKEN'])
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)

!kaggle datasets download -d balraj98/deepglobe-land-cover-classification-dataset -p ./deepglobe --unzip
print('✅ DeepGlobe dataset downloaded')

 29%|███████████▏                           | 801M/2.74G [00:53<02:14, 15.6MB/s]^C
✅ DeepGlobe dataset downloaded


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 3: Convert DeepGlobe Masks → YOLO-Seg     ║
# ║  4-Class: Trees, Parks, Water, Roads             ║
# ╚══════════════════════════════════════════════════╝
import cv2
import numpy as np
import glob
import os
import random
from tqdm import tqdm

# ── DeepGlobe RGB → Our Class Mapping ──
# Each entry: (R, G, B) → class_id
CLASS_COLORS = {
    # Class 0: Trees & Green Cover (DeepGlobe "Forest")
    (0, 255, 0): 0,
    # Class 1: Parks & Open Spaces (DeepGlobe "Agriculture" + "Rangeland")
    (255, 255, 0): 1,    # Agriculture = yellow
    (255, 0, 255): 1,    # Rangeland = magenta
    # Class 2: Water Bodies (DeepGlobe "Water")
    (0, 0, 255): 2,
    # Class 3: Roads & Footpaths (DeepGlobe "Barren")
    (255, 255, 255): 3,
    # SKIP: Urban (0,255,255) → handled by separate building model
    # SKIP: Unknown (0,0,0) → background
}

CLASS_NAMES = {
    0: 'Trees & Green Cover',
    1: 'Parks & Open Spaces',
    2: 'Water Bodies',
    3: 'Roads & Footpaths',
}

TOLERANCE = 25  # Color matching tolerance (handles JPEG compression artifacts)
MIN_CONTOUR_AREA = 300  # Minimum polygon area in pixels

# ── Create YOLO directory structure ──
for split in ['train', 'val']:
    os.makedirs(f'landcover_yolo/images/{split}', exist_ok=True)
    os.makedirs(f'landcover_yolo/labels/{split}', exist_ok=True)

# ── Find all mask files ──
mask_files = glob.glob('./deepglobe/**/*_mask.png', recursive=True)
print(f'Found {len(mask_files)} mask files')

# ── Shuffle and split 85/15 ──
random.seed(42)  # Reproducible splits
random.shuffle(mask_files)
split_idx = int(len(mask_files) * 0.85)

converted = 0
skipped = 0
class_counts = {0: 0, 1: 0, 2: 0, 3: 0}

for i, mask_path in enumerate(tqdm(mask_files, desc='Converting')):
    split = 'train' if i < split_idx else 'val'
    basename = os.path.basename(mask_path).replace('_mask.png', '')

    # Find corresponding satellite image
    img_path = mask_path.replace('_mask.png', '_sat.jpg')
    if not os.path.exists(img_path):
        img_path = mask_path.replace('_mask.png', '.jpg')
    if not os.path.exists(img_path):
        skipped += 1
        continue

    # Read mask as RGB
    mask_bgr = cv2.imread(mask_path)
    if mask_bgr is None:
        skipped += 1
        continue
    mask_rgb = cv2.cvtColor(mask_bgr, cv2.COLOR_BGR2RGB)
    h, w = mask_rgb.shape[:2]

    # ── Extract polygons for each class ──
    label_lines = []
    for color, class_id in CLASS_COLORS.items():
        # Create binary mask for this color
        diff = np.abs(mask_rgb.astype(np.int16) - np.array(color, dtype=np.int16))
        binary = np.all(diff < TOLERANCE, axis=2).astype(np.uint8) * 255

        # Morphological cleanup to reduce noise
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

        # Find contours
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for contour in contours:
            if cv2.contourArea(contour) < MIN_CONTOUR_AREA:
                continue

            # Simplify polygon (reduce points for efficiency)
            epsilon = 0.004 * cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, epsilon, True)
            if len(approx) < 3:
                continue

            # Normalize coordinates to [0, 1]
            points = approx.reshape(-1, 2)
            norm_pts = []
            for px, py in points:
                nx = max(0.0, min(1.0, px / w))
                ny = max(0.0, min(1.0, py / h))
                norm_pts.extend([f'{nx:.6f}', f'{ny:.6f}'])
            label_lines.append(f"{class_id} {' '.join(norm_pts)}")
            class_counts[class_id] += 1

    # Only save images that have at least one valid polygon
    if not label_lines:
        skipped += 1
        continue

    # Save satellite image
    img = cv2.imread(img_path)
    cv2.imwrite(f'landcover_yolo/images/{split}/{basename}.jpg', img)

    # Save YOLO label file
    with open(f'landcover_yolo/labels/{split}/{basename}.txt', 'w') as f:
        f.write('\n'.join(label_lines) + '\n')
    converted += 1

# ── Print dataset statistics ──
train_count = len(os.listdir('landcover_yolo/images/train'))
val_count = len(os.listdir('landcover_yolo/images/val'))

print(f'\n{"═" * 55}')
print(f'  📊 DATASET CONVERSION COMPLETE')
print(f'{"═" * 55}')
print(f'  Images converted: {converted} | Skipped: {skipped}')
print(f'  Train: {train_count} | Val: {val_count}')
print(f'  ─────────────────────────────')
for cid, name in CLASS_NAMES.items():
    print(f'  Class {cid} ({name}): {class_counts[cid]} polygons')
print(f'{"═" * 55}')

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 4: Write data.yaml                        ║
# ╚══════════════════════════════════════════════════╝
import os

yaml_content = f"""path: {os.path.abspath('landcover_yolo')}
train: images/train
val: images/val

nc: 4
names:
  0: Trees & Green Cover
  1: Parks & Open Spaces
  2: Water Bodies
  3: Roads & Footpaths
"""

with open('landcover_yolo/data.yaml', 'w') as f:
    f.write(yaml_content)

print('✅ data.yaml written:')
print(yaml_content)

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 5: TRAIN — Maximum Accuracy Configuration ║
# ╚══════════════════════════════════════════════════╝
from ultralytics import YOLO

# Load YOLOv11 Medium Segmentation (best accuracy/speed balance)
model = YOLO('yolo11m-seg.pt')

print('🚀 Starting training — optimized for satellite imagery accuracy...')
print('   This will take ~100 minutes on T4 GPU.')
print('   Early stopping will auto-halt if accuracy plateaus.\n')

results = model.train(
    # ── Data ──
    data='landcover_yolo/data.yaml',
    imgsz=640,
    batch=16,
    device='0',

    # ── Training schedule ──
    epochs=100,
    patience=20,           # Stop if no improvement for 20 epochs
    save_period=10,        # Checkpoint every 10 epochs
    val=True,

    # ── Optimizer ──
    optimizer='AdamW',
    lr0=0.001,             # Initial learning rate
    lrf=0.01,              # Final LR = lr0 * lrf
    weight_decay=0.0005,
    warmup_epochs=5,       # Gradual warmup

    # ── Augmentations (critical for satellite imagery) ──
    augment=True,
    degrees=180.0,         # Full rotation — satellite has no "up"
    flipud=0.5,            # Vertical flip
    fliplr=0.5,            # Horizontal flip
    mosaic=1.0,            # Mosaic augmentation (4 images combined)
    mixup=0.15,            # MixUp augmentation
    copy_paste=0.3,        # Copy-paste augmentation for segmentation
    hsv_h=0.015,           # Hue jitter (handles lighting variation)
    hsv_s=0.7,             # Saturation jitter
    hsv_v=0.4,             # Value/brightness jitter
    scale=0.5,             # Random scale ±50%
    translate=0.1,         # Random translate ±10%
    perspective=0.001,     # Slight perspective distortion

    # ── Output ──
    project='runs/train',
    name='landcover_v1',
    exist_ok=True,
)

print('\n✅ Training complete!')

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 6: Evaluate Accuracy                      ║
# ╚══════════════════════════════════════════════════╝
from ultralytics import YOLO

best = YOLO('runs/train/landcover_v1/weights/best.pt')
metrics = best.val(data='landcover_yolo/data.yaml', device='0')

CLASS_NAMES = ['Trees & Green Cover', 'Parks & Open Spaces', 'Water Bodies', 'Roads & Footpaths']

print('\n' + '═' * 60)
print('  📊 ACCURACY REPORT — Land Cover Segmentation')
print('═' * 60)
print(f'  Overall Seg Precision:     {metrics.seg.mp:.4f}')
print(f'  Overall Seg Recall:        {metrics.seg.mr:.4f}')
print(f'  Overall Seg mAP@50:        {metrics.seg.map50:.4f}')
print(f'  Overall Seg mAP@50-95:     {metrics.seg.map:.4f}')
print('─' * 60)

# Per-class breakdown
if hasattr(metrics.seg, 'ap_class_index') and metrics.seg.ap_class_index is not None:
    for i, cls_idx in enumerate(metrics.seg.ap_class_index):
        name = CLASS_NAMES[cls_idx] if cls_idx < len(CLASS_NAMES) else f'class_{cls_idx}'
        ap50 = metrics.seg.ap50[i] if i < len(metrics.seg.ap50) else 0
        print(f'  {name:30s} mAP@50: {ap50:.4f}')

print('═' * 60)
if metrics.seg.map50 > 0.70:
    print('  🌟 EXCELLENT — Production-ready accuracy!')
elif metrics.seg.map50 > 0.50:
    print('  👍 GOOD — Usable for deployment.')
else:
    print('  ⚠️  Consider adding more data or training longer.')
print('═' * 60)

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 7: Download Weights                       ║
# ╚══════════════════════════════════════════════════╝
from google.colab import files

# Rename for clarity
import shutil
shutil.copy('runs/train/landcover_v1/weights/best.pt', 'landcover.pt')

files.download('landcover.pt')
print('\n🎉 Done! Place landcover.pt into your project:')
print('   → backend/models/landcover.pt')
print('   Then restart your FastAPI server.')